# Explore the public capstone inputs

How large are the pinned inputs, and how is the available HbA1c target distributed?

This is the data-exploration step of the GluFormer + OpenEvolve reproducibility audit. Each CSV row represents a recorded observation. Repeated recordings can belong to the same participant.


In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "config/project.json").is_file(), "Run from the project folder."
sys.path.insert(0, str(root / "scripts"))
from project_io import verify_inputs

verify_inputs()
results = root / "results"
results.mkdir(exist_ok=True)
filenames = {
    "Embeddings": "GluFormer_Representations_Shanghai_2023.csv",
    "GMI": "Shanghai_GMI.csv",
    "HbA1c": "Shanghai_results.csv",
}
frames = {
    name: pd.read_csv(root / "data/raw" / filename, index_col=0)
    for name, filename in filenames.items()
}


In [ ]:
# Aggregate diagnostics, without displaying individual records.
summary = pd.DataFrame([
    {
        "Input": name,
        "Rows": len(frame),
        "Data columns": frame.shape[1],
        "Distinct recorded IDs": frame.index.nunique(),
        "Missing cells": int(frame.isna().sum().sum()),
    }
    for name, frame in frames.items()
])
summary


In [ ]:
target = pd.to_numeric(frames["HbA1c"].iloc[:, 0]).dropna()
target_stats = target.rename("HbA1c (mmol/mol)").describe()
exploration = {
    "inputs": summary.to_dict(orient="records"),
    "hba1c": target_stats.to_dict(),
}
(results / "explore_inputs.json").write_text(json.dumps(exploration, indent=2) + "\n")
target_stats.round(2)


In [ ]:
#| label: fig-hba1c-records
#| fig-cap: "HbA1c distribution across available records in the pinned public Shanghai target CSV. Repeated recordings can belong to the same participant."
#| fig-alt: "Histogram of HbA1c records, with HbA1c in mmol/mol on the horizontal axis and the number of records on the vertical axis."
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(target, bins=15, edgecolor="white")
ax.set_xlabel("HbA1c (mmol/mol)")
ax.set_ylabel("Records")
fig.tight_layout()
fig.savefig(results / "hba1c_records.png", dpi=150)
plt.show()


## What carries into the report

The aggregate input table and target histogram are saved in `results/` for the Quarto report. The existing audit documents historical participant-grouping problems, target-scaling problems and repeated use of development folds. These limits remain relevant to interpretation of the model comparison.
